# 1. What is actually in our portfolio?

Your portfolio manager asks: **“We own four stocks. Are we diversified?”**
By the end, you will answer with a small table, a chart, and a sentence you can defend.
Our prices are a **synthetic round-number snapshot** so you can check the arithmetic yourself.

You will practice three Python moves:
1. **Select** the rows and columns that answer a question.
2. **Calculate** a new column without writing a loop over rows.
3. **Combine and summarize** data without losing or multiplying positions.

A position's value is shares × price.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
%matplotlib inline
# Supplied setup: locate this repository even if Jupyter starts in a lecture folder.
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
plt.style.use('seaborn-v0_8-whitegrid')

## 1. From a Python dictionary to a table
Lecture 1 introduced dictionaries and lists. Each dictionary key below becomes a column; each list supplies its values.
**Predict:** How many rows and columns will this table have? What does one row mean?

In [ ]:
positions = pd.DataFrame({
    'ticker': ['AAPL', 'MSFT', 'NVDA', 'JPM'],
    'shares': [20, 5, 20, 10],
    'price': [100.0, 200.0, 50.0, 100.0],
})
positions

In [ ]:
print('Rows, columns:', positions.shape)
print('Column names:', positions.columns.tolist())
positions.info()

A **DataFrame** is the table. A **Series** is one labeled column.
The index on the left identifies rows. It is separate from the ticker column.
Run the next cell twice: `positions` has not changed. A selection alone does not update the table.

In [ ]:
prices = positions['price']
print(type(prices))
display(prices)
display(positions[['ticker', 'price']])  # A list of column names keeps the result a table.

## 2. Select the observations you mean
Three patterns will cover most of today's selection:

| Intent | Pattern |
|---|---|
| Select columns | `table[['column_a', 'column_b']]` |
| Select by position | `table.iloc[:2]` |
| Select matching rows and columns | `table.loc[condition, ['column_a']]` |

**Predict:** What are the first two rows after sorting by price? Is the highest share price necessarily our largest position?

In [ ]:
by_price = positions.sort_values('price', ascending=False)
display(by_price)
print('First row by position:')
display(by_price.iloc[0])
print('Original row labeled 0:')
display(by_price.loc[0])

Sorting moves rows, but labels travel with them. `.iloc[0]` means first position; `.loc[0]` means label 0.
This difference explains many “my code runs but returns the wrong row” bugs.

### Worked example: select higher-priced shares
A Boolean mask has one value per row. Here we keep shares priced at least $150 and display the ticker and price.

In [ ]:
example_price_mask = positions['price'] >= 150
display(example_price_mask)
example_selection = positions.loc[example_price_mask, ['ticker', 'price']]
display(example_selection)

### Read an error without panicking
`positions['Price']` would raise `KeyError`: our column is named `price`.
Before asking for help, read the last line of the error and inspect `positions.columns`.
To combine conditions, use `(condition_a) & (condition_b)`; Python's `and` expects one truth value, not a Series of them.

In [ ]:
mask = (positions['price'] >= 100) & (positions['shares'] < 20)
display(mask)
display(positions.loc[mask, ['ticker', 'shares', 'price']])

## 3. Calculate dollars, then weights
**On paper first:** calculate AAPL's dollar value and the total portfolio value.
Now predict AAPL's weight. It should be **40%**, even though AAPL is only one of four names.

A column calculation is **vectorized**: the multiplication acts on every row. The scalar total is shared across all rows.

In [ ]:
portfolio = positions.copy()
portfolio['position_value'] = portfolio['shares'] * portfolio['price']
total_value = portfolio['position_value'].sum()
portfolio['weight_pct'] = portfolio['position_value'] / total_value * 100
display(portfolio)
print('Portfolio value:', total_value)
print('Weights sum to:', portfolio['weight_pct'].sum())

### Worked example: add five JPM shares
Increase JPM from 10 to 15 shares on a copy. The extra $500 raises the portfolio total to $5,500, so AAPL's unchanged $2,000 position now has a smaller weight.

In [ ]:
example_rebalanced = positions.copy()
example_rebalanced.loc[example_rebalanced['ticker'] == 'JPM', 'shares'] = 15
example_rebalanced['position_value'] = example_rebalanced['shares'] * example_rebalanced['price']
example_rebalanced['weight_pct'] = example_rebalanced['position_value'] / example_rebalanced['position_value'].sum() * 100
display(example_rebalanced)
print('AAPL weight (%):', example_rebalanced.set_index('ticker').loc['AAPL', 'weight_pct'])

### Labels help keep calculations aligned
A Series assignment matches index labels. Set ticker as the index when that is the identity you need.
**Predict:** The prices below are in a different order. Will AAPL receive JPM's price?

In [ ]:
indexed_shares = positions.set_index('ticker')['shares']
reordered_prices = positions.set_index('ticker')['price'].sort_index(ascending=False)
display(indexed_shares * reordered_prices)
# Multiplying plain lists/arrays would discard these ticker labels.

## 4. Add sector information and summarize
Our holdings table knows prices; another table knows sectors.
A **left merge** keeps each holding and looks up its matching ticker. It matches keys, not row positions.
`validate='one_to_one'` is a useful check when each ticker should appear once in each table.

In [ ]:
sectors = pd.DataFrame({
    'ticker': ['JPM', 'NVDA', 'AAPL', 'MSFT'],
    'sector': ['Financials', 'Technology', 'Technology', 'Technology'],
})
portfolio = portfolio.merge(sectors, on='ticker', how='left', validate='one_to_one')
display(portfolio)
assert portfolio['sector'].notna().all(), 'Some tickers were not matched.'

`groupby` means “do this calculation separately for each group.”
Read the next expression from left to right: group by sector → choose position values → sum.
**Predict:** Is technology 75% or 80% of the portfolio? Explain why counting names gives a different answer.

In [ ]:
sector_values = portfolio.groupby('sector')['position_value'].sum()
sector_weights = sector_values / sector_values.sum() * 100
display(sector_weights)
ax = sector_weights.plot.bar(rot=0, title='Where are our dollars?', ylabel='Portfolio weight (%)')
plt.show()

### A small bug with a large effect
If the right table contains two AAPL rows, a merge could duplicate the AAPL position.
We deliberately catch that problem below. This is the only new error-handling syntax here: `try` attempts the operation; `except` explains an expected error.
**Discuss:** why would dropping an arbitrary duplicate be a business decision as well as a coding decision?

In [ ]:
duplicate_sectors = pd.concat([sectors, sectors.loc[sectors['ticker'] == 'AAPL']], ignore_index=True)
try:
    portfolio[['ticker', 'position_value']].merge(duplicate_sectors, on='ticker', validate='one_to_one')
except pd.errors.MergeError:
    print('Duplicate ticker in the lookup: resolve it before trusting the dollar totals.')

## 5. Turn a selection into a desk update
Find Technology positions worth at least $1,500. Then report the Financials allocation. A useful summary names the quantity and its unit.

In [ ]:
example_desk = portfolio.loc[(portfolio['sector'] == 'Technology') & (portfolio['position_value'] >= 1500),
                              ['ticker', 'position_value']]
example_financials_weight = sector_weights.loc['Financials']
display(example_desk)
print(f'Financials represents {example_financials_weight:.0f}% of portfolio value.')

## Exit check
Without copying code: how would you (a) select by a condition, (b) add a calculated column, and (c) summarize by a category?
If one of those still feels unclear, circle it and use the matching worked example when reviewing the lesson or during office hours.

Next: [Can we trust this price chart?](3_visualize_and_clean.ipynb).
For method variants, use the [Series](reference/2_pandas_series_basics.ipynb) and [DataFrame](reference/3_pandas_dataframes_basics.ipynb) references after class.

## Exercises
Use the tables and functions above to complete these tasks. Replace the placeholders with your code, then run the checks.

### Your turn A: a trading-desk request
Select `ticker` and `shares` for positions with **at least 10 shares**. Assign the result to `your_selection`.
**Hint:** build a Boolean mask with `positions['shares'] >= 10`, then pass it to `.loc`.
**Stretch:** also require price below 150. Use `&` and parentheses around each comparison.

In [ ]:
your_selection = None  # Replace None with your selection.
# Example shape: positions.loc[condition, ['ticker', 'shares']]

In [ ]:
if your_selection is None:
    print('Complete A, then run this check.')
else:
    assert isinstance(your_selection, pd.DataFrame), 'Select a list of columns to keep a DataFrame.'
    assert your_selection.columns.tolist() == ['ticker', 'shares']
    assert your_selection['ticker'].tolist() == ['AAPL', 'NVDA', 'JPM'], 'Check >= versus >.'
    assert your_selection['shares'].tolist() == [20, 20, 10]
    print('A passed. Explain what the mask does.')

### Your turn B: change the position
The desk doubles its MSFT shares from 5 to 10. What happens to **AAPL's** weight?
Make the change on a copy, recompute values and weights, and assign that DataFrame to `your_rebalanced`.
**Hint:** `changed.loc[changed['ticker'] == 'MSFT', 'shares'] = 10`.
**Stretch:** explain why doubling one holding does not double its portfolio weight.

In [ ]:
your_rebalanced = None
# changed = positions.copy()
# Update one row with .loc, then calculate position_value and weight_pct.
# your_rebalanced = changed

In [ ]:
if your_rebalanced is None:
    print('Complete B, then check the new portfolio total and AAPL weight.')
else:
    assert np.isclose(your_rebalanced['position_value'].sum(), 6000)
    assert np.isclose(your_rebalanced.set_index('ticker').loc['AAPL', 'weight_pct'], 100 / 3)
    assert np.isclose(your_rebalanced['weight_pct'].sum(), 100)
    assert positions.set_index('ticker').loc['MSFT', 'shares'] == 5, 'Keep the original table unchanged.'
    print('B passed. Explain the new denominator.')

### Your turn: prepare a short desk update
Use the **original** portfolio, not the doubled-MSFT scenario.

1. Find positions worth at least $1,000 **outside Technology**; keep ticker and position value.
2. Report the largest sector weight.
3. Write one sentence answering the opening question, using a number from your analysis.

Assign the table to `your_desk_table` and the percentage to `your_sector_weight`.
**Hint:** borrow the selection pattern, not an entire solution.
**Stretch:** add one new holding with `pd.concat`; identify which lookup and calculation steps must run again.

Vary one input and explain how it changes the result.

In [ ]:
your_desk_table = None
your_sector_weight = None
your_conclusion = ''  # Explain a concentration, not a buy/sell recommendation.

In [ ]:
if your_desk_table is None or your_sector_weight is None:
    print('Finish the desk update: one table, one percentage, one sentence.')
else:
    assert your_desk_table['ticker'].tolist() == ['JPM']
    assert your_desk_table['position_value'].tolist() == [1000]
    assert np.isclose(your_sector_weight, 80)
    assert your_conclusion.strip(), 'Explain the number in your own words.'
    print('Desk update checks passed. Check that your sentence states the units.')